### Library

In [1]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)
print(f"Project root added: {project_root}")

Project root added: /home/hhuscout/myproject/deepkriging-sphere


In [2]:
import multiprocessing as mp
if mp.get_start_method(allow_none=True) is None:
    mp.set_start_method('spawn')

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', message='.*input_shape.*')
warnings.filterwarnings('ignore', message='.*structure of.*inputs.*')

import os, time, gc
from types import SimpleNamespace

import numpy as np
import pandas as pd
from scipy.special import gamma

import jax, jax.numpy as jnp

import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber
from tensorflow.keras import backend as K

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder

import optuna
import plotly.io as pio
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature

2026-02-27 08:31:12.357643: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772152272.368359   23958 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772152272.372214   23958 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772152272.381247   23958 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772152272.381263   23958 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772152272.381264   23958 computation_placer.cc:177] computation placer alr

### Environment Setting

In [3]:
np_f32 = np.float32
jnp_f32 = jnp.float32
dtype_basis = np.float32

jax.config.update("jax_enable_x64", False)

pio.renderers.default = "notebook"
warnings.filterwarnings("ignore", category=UserWarning)

os.environ.update({"TF_CPP_MIN_LOG_LEVEL": "2"})
optuna.logging.set_verbosity(optuna.logging.WARNING)

os.environ.setdefault("OMP_NUM_THREADS", "12")
os.environ.setdefault("MKL_NUM_THREADS", "12")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "12")

def init_hardware(dtype: str = "float32"):
    gpus = tf.config.list_physical_devices("GPU")
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    strategy = (tf.distribute.MirroredStrategy() if len(gpus) > 1 else tf.distribute.get_strategy())
    mixed_precision.set_global_policy(dtype)
    return strategy

strategy = init_hardware(dtype="float32")

### Our Functions

In [4]:
from spherical_deepkriging.basis_functions.wendland.wenland import wendland
from spherical_deepkriging.basis_functions.mrts.mrts import mrts0

from spherical_deepkriging.models.deep_kriging import DeepKrigingTrainer, DeepKrigingDefaultTrainer
from spherical_deepkriging.configs import DeepKrigingModelConfig, DeepKrigingDefaultConfig
from spherical_deepkriging.models.universal_kriging import UniversalKriging

from rpy2.robjects.conversion import localconverter
from spherical_deepkriging.basis_functions.mrts_sphere.sphere import mrts_sphere, numpy2ri_converter

R callback write-console: Registered S3 methods overwritten by 'RcppEigen':
  method               from         
  predict.fastLm       RcppArmadillo
  print.fastLm         RcppArmadillo
  summary.fastLm       RcppArmadillo
  print.summary.fastLm RcppArmadillo
  


### Helper

In [5]:
def data_preprocessing(data_path, variable, num_sample, seed):
    data = (pd.read_csv(data_path).sample(n=num_sample, random_state=seed).replace("-", np.nan).dropna())

    numeric_cols = ["longitude", "latitude", variable]
    data[numeric_cols] = data[numeric_cols].apply(pd.to_numeric, errors="coerce")
    data.dropna(subset=numeric_cols, inplace=True)

    lon, lat = data["longitude"].to_numpy(), data["latitude"].to_numpy()

    norm_lon = (lon - lon.min()) / (lon.max() - lon.min())
    norm_lat = (lat - lat.min()) / (lat.max() - lat.min())

    location_data = np.column_stack([lat, lon]).astype("float32")
    location_data_norm = np.column_stack([norm_lon, norm_lat]).astype("float32")
    y_combined = data[variable].to_numpy().astype("float32")[:, None]

    # Handle
    categorical_data = None

    return location_data, location_data_norm, categorical_data, y_combined


def precompute_wendland(location, num_basis):
    parts = []
    for nb in num_basis:
        grid = np.column_stack(np.meshgrid(
            np.linspace(0, 1, int(np.sqrt(nb)), dtype=np_f32),
            np.linspace(0, 1, int(np.sqrt(nb)), dtype=np_f32),
        )).reshape(-1, 2).astype(np_f32)

        theta = np_f32(2.5)/np_f32(np.sqrt(nb))
        parts.append(
            wendland(location, grid, theta=theta, k=2)
        )

    return np.hstack(parts).astype(dtype_basis, copy=False)


def precompute_max_mrts(distance_type, location_data, knot_num, order_max, knot=None):
    if knot is None:
        idx_knot = np.random.choice(location_data.shape[0], knot_num, replace=False)
        knot = location_data[idx_knot].astype(np_f32)
    else:
        idx_knot = None

    if distance_type == "sphere":
        with localconverter(numpy2ri_converter):
            res_r = mrts_sphere(knot, order_max, location_data.astype(np_f32))
        res_dict = dict(zip(res_r.names(), res_r))
        phi = np.asarray(res_dict["mrts"], dtype=dtype_basis)
    else:
        phi = np.asarray(
            mrts0(jnp.asarray(knot, dtype=jnp_f32), k=order_max, 
                  x=jnp.asarray(location_data, dtype=jnp_f32)), dtype=dtype_basis
        )

    return phi, idx_knot, knot


def prepare_data(categorical_data, basis, y_combined, seed, split_ratio=(0.8, 0.1, 0.1)):
    idx_all = np.arange(basis.shape[0])
    train_ratio, val_ratio, test_ratio = split_ratio
    
    train_val_x1, test_x1 = train_test_split(
        idx_all, train_size=train_ratio+val_ratio, random_state=seed)
    train_x1, val_x1 = train_test_split(
        train_val_x1, train_size=train_ratio/(train_ratio+val_ratio), random_state=seed)
    
    X_train_cont, X_val_cont, X_test_cont = (
        basis[train_x1], basis[val_x1], basis[test_x1])
    y_train, y_val, y_test = (
        y_combined[train_x1], y_combined[val_x1], y_combined[test_x1])
    
    def flatten(targets):
        return targets.reshape(-1).astype(np_f32, copy=False)
    y_train_flat, y_val_flat, y_test_flat = map(flatten, [y_train, y_val, y_test])

    def flatten(covariates):
        return covariates.reshape(-1, basis.shape[1]).astype(np_f32)
    X_train_cont_flat, X_val_cont_flat, X_test_cont_flat = map(flatten, [X_train_cont, X_val_cont, X_test_cont])
    
    # Handle categorical features
    if categorical_data is None:
        zero_vector = lambda n: np.zeros((n, 0), dtype=np_f32)
        X_train_cat, X_val_cat, X_test_cat = map(zero_vector, [len(X_train_cont_flat), len(X_val_cont_flat), len(X_test_cont_flat)])
    else:
        cat_train = categorical_data.reshape(-1, 1)[train_x1]
        cat_val = categorical_data.reshape(-1, 1)[val_x1]
        cat_test = categorical_data.reshape(-1, 1)[test_x1]
        
        OHE = OneHotEncoder(sparse_output=False, dtype=np_f32)
        X_train_cat = OHE.fit_transform(cat_train).astype(np_f32)
        X_val_cat = OHE.transform(cat_val).astype(np_f32)
        X_test_cat = OHE.transform(cat_test).astype(np_f32)
    
    return (X_train_cont_flat, X_train_cat, y_train_flat,
            X_val_cont_flat, X_val_cat, y_val_flat,
            X_test_cont_flat, X_test_cat, y_test_flat)


def train_eval(name_model, epochs, batch_size, loss, dropout_rate,
               X_train, X_train_cat, y_train,
               X_val, X_val_cat, y_val,
               X_test, X_test_cat, y_test):

    if name_model in ["OLS_wendland", "OLS_sphere"]:
        t0 = time.time()
        model = LinearRegression().fit(X_train, y_train)

        val_loss = float(mean_squared_error(y_val, model.predict(X_val)))
        y_pred = model.predict(X_test).astype(np_f32).reshape(-1)
        training_time = time.time() - t0
        
        metrics = {
            "Model": name_model,
            "Val_loss": float(val_loss),
            "MSPE": float(mean_squared_error(y_test, y_pred)),
            "RMSE": float(np.sqrt(mean_squared_error(y_test, y_pred))),
            "MAE": float(mean_absolute_error(y_test, y_pred)),
            "R2": float(r2_score(y_test, y_pred)),
            "Time": float(training_time),
        }

        return metrics, model
    
    elif name_model == "DeepKriging_wendland":
        config = DeepKrigingDefaultConfig(
            input_dim=X_train.shape[1],
            output_type='continuous',
            optimizer=Adam(learning_rate=1e-3),
            loss=loss,
            epochs=epochs,
            batch_size=batch_size,
            verbose=0
        )

    elif name_model == "DeepKriging_wendland*":
        optimizer = Adam(learning_rate=5e-3)
        config = DeepKrigingModelConfig(
            input_dim=X_train.shape[1],
            output_type='continuous',
            hidden_layers=[1024, 512, 256, 128, 64],
            activation='relu',
            dropout_rate=dropout_rate,
            optimizer=optimizer,
            loss=loss,
            metrics=['mae'],
            epochs=epochs,
            batch_size=batch_size,
            patience=40,
            verbose=0
        )

    elif name_model in ["DeepKriging_mrts", "DeepKriging_sphere", "DeepKriging_sphere_Huber"]:
        optimizer = Adam(learning_rate=5e-3)
        config = DeepKrigingModelConfig(
            input_dim=X_train.shape[1],
            output_type='continuous',
            hidden_layers=[1024, 512, 256, 128, 64],
            activation='relu',
            dropout_rate=dropout_rate,
            optimizer=optimizer,
            loss=loss,
            metrics=['mae'],
            epochs=epochs,
            batch_size=batch_size,
            patience=40,
            verbose=0
        )

    t0 = time.time()
    with strategy.scope():
        if name_model == "DeepKriging_wendland":
            model = DeepKrigingDefaultTrainer(config)
        elif name_model == "DeepKriging_wendland*":
            model = DeepKrigingTrainer(config)
        else:
            model = DeepKrigingTrainer(config)
        model.model.compile(optimizer=config.optimizer, loss=config.loss, metrics=config.metrics)

    checkpoint_path = f"best_{name_model}_{time.time_ns()}.weights.h5"
    if name_model == "DeepKriging_wendland":
        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(
                filepath=checkpoint_path, monitor="val_loss", mode="min",
                save_best_only=True, save_weights_only=True, verbose=0)
        ]
    else:
        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(
                filepath=checkpoint_path, monitor="val_loss", mode="min",
                save_best_only=True, save_weights_only=True, verbose=0),
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=config.patience, restore_best_weights=True, verbose=0),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=max(5, config.patience // 2), min_lr=1e-6, verbose=0)
        ]

    train_dataset = tf.data.Dataset.from_tensor_slices((
        (X_train, X_train_cat), y_train
    )).batch(config.batch_size).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((
        (X_val, X_val_cat), y_val
    )).batch(config.batch_size).prefetch(tf.data.AUTOTUNE)

    history = model.model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=epochs,
        callbacks=callbacks,
        verbose=0
    )

    if os.path.exists(checkpoint_path):
        model.model.load_weights(checkpoint_path)
        os.remove(checkpoint_path)
    
    val_loss = float(np.min(history.history["val_loss"]))
    y_pred = model.model.predict([X_test, X_test_cat], verbose=0).reshape(-1).astype(np_f32)
    training_time = time.time() - t0

    metrics = {
        "Model": name_model,
        "Val_loss": float(val_loss),
        "MSPE": float(mean_squared_error(y_test, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_test, y_pred))),
        "MAE": float(mean_absolute_error(y_test, y_pred)),
        "R2": float(r2_score(y_test, y_pred)),
        "Time": float(training_time),
    }

    del train_dataset, val_dataset
    K.clear_session()
    gc.collect()
    
    return metrics, model

### Plot Helper

In [6]:
def plot_robinson_subplots(data_dict, vmin, vmax, title_main):
    cmap = mcolors.LinearSegmentedColormap.from_list(
        "teal-yellow-red", ["#00AAAA", "#FFFFBB", "#FF3333"], N=256)
    
    n_plots = len(data_dict)
    n_cols = 2
    n_rows = (n_plots + n_cols - 1) // n_cols
    
    fig = plt.figure(figsize=(20, 6 * n_rows))
    fig.suptitle(title_main, fontsize=20, fontweight='bold', y=0.98)
    
    for idx, (subplot_title, (lon, lat, val)) in enumerate(data_dict.items(), 1):
        ax = fig.add_subplot(n_rows, n_cols, idx, projection=ccrs.Robinson())
        ax.set_global()
        ax.add_feature(cfeature.LAND, facecolor="white", alpha=0.3)
        ax.add_feature(cfeature.OCEAN, facecolor="white", alpha=0.3)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.3, alpha=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.2, alpha=0.5)
        
        scatter = ax.scatter(lon, lat, c=val, cmap=cmap, s=10, transform=ccrs.PlateCarree(), vmin=vmin, vmax=vmax)
        ax.set_title(subplot_title, fontsize=14, pad=10, fontweight='bold')
        
        cbar = plt.colorbar(scatter, ax=ax, orientation='horizontal', pad=0.05, shrink=0.6, aspect=30)
        cbar.set_label('Temperature (°C)', fontsize=10)
    
    plt.tight_layout()
    
    return fig

### Experiment Setup

In [7]:
# Model Setup
seed = 1234
epochs = 500
batch_size = 2048
num_sample = 200000
huber_delta = 1.345

# Data
data_path = "data_speed_combined.csv"

# Basis Setup
# Wendland
num_basis = [10**2, 19**2, 37**2]

knot_num = 2000
order_max = 1400
base_orders = [50, 100, 200, 500, 700, 1000, 1400]

# Set to 2 repeats per variable
repeat_experiment = 2

### Outcome

In [8]:
# Dictionary to hold the final summary table for every hour
all_hours_summary = {}

# # Generate the 24 column names: 'vec' (Hour 1), 'vec.1' (Hour 2), ..., 'vec.23' (Hour 24)
# target_variables = ["vec"] + [f"vec.{i}" for i in range(5, 24)]
# ONLY include hours 5 through 24 (which corresponds to vec.4 through vec.23)
target_variables = [f"vec.{i}" for i in range(4, 24)]

for hour, variable in enumerate(target_variables, start=5):
    
    print(f"\n\n{'#'*80}")
    print(f"🚀 STARTING DATASET: Hour {hour:02d} | Target Column: '{variable}'")
    print(f"{'#'*80}")
    
    best_orders = {}
    experiment_results = {
        model: {"MSPE": [], "RMSE": [], "MAE": [], "R2": []}
        for model in ["OLS_wendland", "OLS_sphere", "DeepKriging_wendland", "DeepKriging_wendland*", "DeepKriging_mrts", "DeepKriging_sphere", "DeepKriging_sphere_Huber", "UniversalKriging"]
    }

    for repeat in range(repeat_experiment):
        seed_current = seed + repeat

        print(f"\n{'='*80}")
        print(f"🏃 Hour {hour:02d} ('{variable}') - Repeat {repeat+1}/{repeat_experiment}, Seed={seed_current}")
        print(f"{'='*80}")

        # Data Preprocessing (Variable gets dynamically passed in here!)
        location_data, location_data_norm, categorical_data, y_combined = data_preprocessing(data_path, variable, num_sample, seed_current)

        # Compute Basis Functions
        max_Phi_sphere, idx_knot, knot = precompute_max_mrts("sphere", location_data, knot_num, order_max, knot=None)
        max_Phi_sphere = max_Phi_sphere.astype(dtype_basis, copy=False)

        max_Phi_mrts, idx_knot_mrts, knot_mrts = precompute_max_mrts("mrts", location_data, knot_num, order_max, knot=location_data[idx_knot])
        max_Phi_mrts = max_Phi_mrts.astype(dtype_basis, copy=False)

        Phi_wendland = precompute_wendland(location_data_norm, num_basis)

        if repeat == 0:
            # Tuning order parameter for OLS_sphere
            Best_val_OLS_S = float('inf')
            Best_order_OLS_S = None
            Results_order_OLS_S = []
            
            print("\nTuning order parameter for OLS_sphere")
            for order in base_orders:
                Phi_sphere = max_Phi_sphere[:, :order].astype(np_f32)
                parts = prepare_data(categorical_data, Phi_sphere, y_combined, seed_current)
                metrics, model = train_eval("OLS_sphere", None, None, None, None, *parts)
                
                Results_order_OLS_S.append({'order': order, 'val_loss': metrics["Val_loss"], 'mspe': metrics["MSPE"]})
                
                if metrics["Val_loss"] < Best_val_OLS_S:
                    Best_val_OLS_S = metrics["Val_loss"]
                    Best_order_OLS_S = order
                
                del Phi_sphere, parts, model; gc.collect()

            print(f"   {'Order':<10} {'Val Loss':<12} {'Test MSE':<12}")
            print(f"   {'-'*34}")
            for res in Results_order_OLS_S:
                marker = " *" if res['order'] == Best_order_OLS_S else ""
                print(f"   {res['order']:<10} {res['val_loss']:<12.4f} {res['mspe']:<12.4f}{marker}")
            print(f"   Best order: {Best_order_OLS_S}")
            best_orders['OLS_sphere'] = Best_order_OLS_S

            # Tuning order parameter for DeepKriging_mrts
            Best_val_DK_mrts = float('inf')
            Best_order_DK_mrts = None
            Results_order_DK_mrts = []

            print("\nTuning order parameter for DeepKriging_mrts")
            for order in base_orders:
                Phi_mrts = max_Phi_mrts[:, :order].astype(np_f32)
                parts = prepare_data(categorical_data, Phi_mrts, y_combined, seed_current)
                with strategy.scope():
                    metrics, model = train_eval("DeepKriging_mrts", epochs, batch_size, "mse", 0.01, *parts)
                
                Results_order_DK_mrts.append({'order': order, 'val_loss': metrics["Val_loss"], 'mspe': metrics["MSPE"]})
                
                if metrics["Val_loss"] < Best_val_DK_mrts:
                    Best_val_DK_mrts = metrics["Val_loss"]
                    Best_order_DK_mrts = order
                
                del Phi_mrts, parts, model; K.clear_session(); gc.collect()

            print(f"   {'Order':<10} {'Val Loss':<12} {'Test MSE':<12}")
            print(f"   {'-'*34}")
            for res in Results_order_DK_mrts:
                marker = " *" if res['order'] == Best_order_DK_mrts else ""
                print(f"   {res['order']:<10} {res['val_loss']:<12.4f} {res['mspe']:<12.4f}{marker}")
            print(f"   Best order: {Best_order_DK_mrts}")
            best_orders['DeepKriging_mrts'] = Best_order_DK_mrts

            # Tuning order parameter for DeepKriging_sphere
            Best_val_DK_S = float('inf')
            Best_order_DK_S = None
            Results_order_DK_S = []

            print("\nTuning order parameter for DeepKriging_sphere")
            for order in base_orders:
                Phi_sphere = max_Phi_sphere[:, :order].astype(np_f32)
                parts = prepare_data(categorical_data, Phi_sphere, y_combined, seed_current)
                with strategy.scope():
                    metrics, model = train_eval("DeepKriging_sphere", epochs, batch_size, "mse", 0.01, *parts)
                
                Results_order_DK_S.append({'order': order, 'val_loss': metrics["Val_loss"], 'mspe': metrics["MSPE"]})
                
                if metrics["Val_loss"] < Best_val_DK_S:
                    Best_val_DK_S = metrics["Val_loss"]
                    Best_order_DK_S = order
                
                del Phi_sphere, parts, model; K.clear_session(); gc.collect()

            print(f"   {'Order':<10} {'Val Loss':<12} {'Test MSE':<12}")
            print(f"   {'-'*34}")
            for res in Results_order_DK_S:
                marker = " *" if res['order'] == Best_order_DK_S else ""
                print(f"   {res['order']:<10} {res['val_loss']:<12.4f} {res['mspe']:<12.4f}{marker}")
            print(f"   Best order: {Best_order_DK_S}")
            best_orders['DeepKriging_sphere'] = Best_order_DK_S

            # Tuning order parameter for DeepKriging_sphere_Huber
            Best_val_DK_S_H = float('inf')
            Best_order_DK_S_H = None
            Results_order_DK_S_H = []
            
            print("\nTuning order parameter for DeepKriging_sphere_Huber")
            for order in base_orders:
                Phi_sphere = max_Phi_sphere[:, :order].astype(np_f32)
                parts = prepare_data(categorical_data, Phi_sphere, y_combined, seed_current)
                with strategy.scope():
                    metrics, model = train_eval("DeepKriging_sphere_Huber", epochs, batch_size, Huber(delta=huber_delta), 0.01, *parts)
                
                Results_order_DK_S_H.append({'order': order, 'val_loss': metrics["Val_loss"], 'mspe': metrics["MSPE"]})
                
                if metrics["Val_loss"] < Best_val_DK_S_H:
                    Best_val_DK_S_H = metrics["Val_loss"]
                    Best_order_DK_S_H = order
                
                del Phi_sphere, parts, model; K.clear_session(); gc.collect()

            print(f"   {'Order':<10} {'Val Loss':<12} {'Test MSE':<12}")
            print(f"   {'-'*34}")
            for res in Results_order_DK_S_H:
                marker = " *" if res['order'] == Best_order_DK_S_H else ""
                print(f"   {res['order']:<10} {res['val_loss']:<12.4f} {res['mspe']:<12.4f}{marker}")
            print(f"   Best order: {Best_order_DK_S_H}")
            best_orders['DeepKriging_sphere_Huber'] = Best_order_DK_S_H

            # Tuning order parameter for UniversalKriging
            Best_val_UK = float('inf')
            Best_order_UK = None
            Results_order_UK = []
            
            print("\nTuning order parameter for UniversalKriging")
            for order in base_orders:
                Phi_sphere = max_Phi_sphere[:, :order].astype(np_f32)
                
                idx_all = np.arange(Phi_sphere.shape[0])
                train_val_idx, test_idx = train_test_split(idx_all, train_size=0.9, random_state=seed_current)
                train_idx, val_idx = train_test_split(train_val_idx, train_size=8/9, random_state=seed_current)
                
                coords_train, coords_val, coords_test = location_data[train_idx], location_data[val_idx], location_data[test_idx]
                phi_train, phi_val, phi_test = Phi_sphere[train_idx], Phi_sphere[val_idx], Phi_sphere[test_idx]
                y_train, y_val, y_test = y_combined[train_idx].flatten(), y_combined[val_idx].flatten(), y_combined[test_idx].flatten()
                
                uk_model = UniversalKriging(num_neighbors=30, cov_function='exponential')
                uk_model.fit(coords_train, phi_train, y_train, center_y=True)
                
                y_pred_val = uk_model.predict(coords_val, phi_val, return_centered=True)
                val_loss = mean_squared_error(y_val - uk_model.y_mean, y_pred_val)
                
                y_pred_test = uk_model.predict(coords_test, phi_test, return_centered=False)
                test_mse = mean_squared_error(y_test, y_pred_test)
                
                Results_order_UK.append({'order': order, 'val_loss': val_loss, 'mspe': test_mse})
                
                if val_loss < Best_val_UK:
                    Best_val_UK = val_loss
                    Best_order_UK = order
                
                uk_model.cleanup()
                del uk_model, Phi_sphere, coords_train, coords_val, coords_test, phi_train, phi_val, phi_test, y_train, y_val, y_test; gc.collect()
                    
            print(f"   {'Order':<10} {'Val Loss':<12} {'Test MSE':<12}")
            print(f"   {'-'*34}")
            for res in Results_order_UK:
                marker = " *" if res['order'] == Best_order_UK else ""
                print(f"   {res['order']:<10} {res['val_loss']:<12.4f} {res['mspe']:<12.4f}{marker}")
            print(f"   Best order: {Best_order_UK}")
            best_orders['UniversalKriging'] = Best_order_UK
            gc.collect()


        # Execute training with best orders
        Record = {}

        # OLS_wendland
        parts = prepare_data(categorical_data, Phi_wendland, y_combined, seed_current)
        metrics, model = train_eval("OLS_wendland", None, None, None, None, *parts)
        Record["OLS_wendland"] = {
            "MSPE": metrics["MSPE"], "RMSE": metrics["RMSE"], "MAE": metrics["MAE"], "R2": metrics["R2"],
            "Time": metrics["Time"], "Param": "--"
        }
        del parts, model; gc.collect()

        # OLS_sphere
        Phi_sphere = max_Phi_sphere[:, :best_orders['OLS_sphere']].astype(np_f32)
        parts = prepare_data(categorical_data, Phi_sphere, y_combined, seed_current)
        metrics, model = train_eval("OLS_sphere", None, None, None, None, *parts)
        Record["OLS_sphere"] = {
            "MSPE": metrics["MSPE"], "RMSE": metrics["RMSE"], "MAE": metrics["MAE"], "R2": metrics["R2"],
            "Time": metrics["Time"], "Param": best_orders['OLS_sphere']
        }
        del Phi_sphere, parts, model; gc.collect()

        # DeepKriging_wendland
        parts = prepare_data(categorical_data, Phi_wendland, y_combined, seed_current)
        with strategy.scope():
            metrics, model = train_eval("DeepKriging_wendland", epochs, batch_size, "mse", None, *parts)
        Record["DeepKriging_wendland"] = {
            "MSPE": metrics["MSPE"], "RMSE": metrics["RMSE"], "MAE": metrics["MAE"], "R2": metrics["R2"],
            "Time": metrics["Time"], "Param": "--"
        }
        del parts, model; K.clear_session(); gc.collect()

        # DeepKriging_wendland*
        parts = prepare_data(categorical_data, Phi_wendland, y_combined, seed_current)
        with strategy.scope():
            metrics, model = train_eval("DeepKriging_wendland*", epochs, batch_size, "mse", 0.01, *parts)
        Record["DeepKriging_wendland*"] = {
            "MSPE": metrics["MSPE"], "RMSE": metrics["RMSE"], "MAE": metrics["MAE"], "R2": metrics["R2"],
            "Time": metrics["Time"], "Param": "--"
        }
        del parts, model; K.clear_session(); gc.collect()

        # DeepKriging_mrts
        Phi_mrts = max_Phi_mrts[:, :best_orders['DeepKriging_mrts']].astype(np_f32)
        parts = prepare_data(categorical_data, Phi_mrts, y_combined, seed_current)
        with strategy.scope():
            metrics, model = train_eval("DeepKriging_mrts", epochs, batch_size, "mse", 0.01, *parts)
        Record["DeepKriging_mrts"] = {
            "MSPE": metrics["MSPE"], "RMSE": metrics["RMSE"], "MAE": metrics["MAE"], "R2": metrics["R2"],
            "Time": metrics["Time"], "Param": best_orders['DeepKriging_mrts']
        }
        del Phi_mrts, parts, model; K.clear_session(); gc.collect()

        # DeepKriging_sphere
        Phi_sphere = max_Phi_sphere[:, :best_orders['DeepKriging_sphere']].astype(np_f32)
        parts = prepare_data(categorical_data, Phi_sphere, y_combined, seed_current)
        with strategy.scope():
            metrics, model = train_eval("DeepKriging_sphere", epochs, batch_size, "mse", 0.01, *parts)
        Record["DeepKriging_sphere"] = {
            "MSPE": metrics["MSPE"], "RMSE": metrics["RMSE"], "MAE": metrics["MAE"], "R2": metrics["R2"],
            "Time": metrics["Time"], "Param": best_orders['DeepKriging_sphere']
        }
        del Phi_sphere, parts, model; K.clear_session(); gc.collect()

        # DeepKriging_sphere_Huber
        Phi_sphere = max_Phi_sphere[:, :best_orders['DeepKriging_sphere_Huber']].astype(np_f32)
        parts = prepare_data(categorical_data, Phi_sphere, y_combined, seed_current)
        with strategy.scope():
            metrics, model = train_eval("DeepKriging_sphere_Huber", epochs, batch_size, Huber(delta=huber_delta), 0.01, *parts)
        Record["DeepKriging_sphere_Huber"] = {
            "MSPE": metrics["MSPE"], "RMSE": metrics["RMSE"], "MAE": metrics["MAE"], "R2": metrics["R2"],
            "Time": metrics["Time"], "Param": best_orders['DeepKriging_sphere_Huber']
        }
        del Phi_sphere, parts, model; K.clear_session(); gc.collect()

        # UniversalKriging
        t0 = time.time()
        Phi_sphere = max_Phi_sphere[:, :best_orders['UniversalKriging']].astype(np_f32)
        
        idx_all = np.arange(Phi_sphere.shape[0])
        train_val_idx, test_idx = train_test_split(idx_all, train_size=0.9, random_state=seed_current)
        train_idx, _ = train_test_split(train_val_idx, train_size=8/9, random_state=seed_current)

        coords_train, coords_test = location_data[train_idx], location_data[test_idx]
        phi_train, phi_test = Phi_sphere[train_idx], Phi_sphere[test_idx]
        y_train, y_test = y_combined[train_idx].flatten(), y_combined[test_idx].flatten()

        uk_model = UniversalKriging(num_neighbors=30, cov_function='exponential')
        uk_model.fit(coords_train, phi_train, y_train, center_y=True)

        y_pred_test = uk_model.predict(coords_test, phi_test, return_centered=False)

        Record["UniversalKriging"] = {
            "MSPE": mean_squared_error(y_test, y_pred_test),
            "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_test)),
            "MAE": mean_absolute_error(y_test, y_pred_test),
            "R2": r2_score(y_test, y_pred_test),
            "Time": time.time() - t0,
            "Param": best_orders['UniversalKriging']
        }

        uk_model.cleanup()
        del uk_model, Phi_sphere, coords_train, coords_test, phi_train, phi_test, y_train, y_test; gc.collect()

        # Print partial result for this repeat
        result_table = []
        for model in ["OLS_wendland", "OLS_sphere", "DeepKriging_wendland", "DeepKriging_wendland*", "DeepKriging_mrts", "DeepKriging_sphere", "DeepKriging_sphere_Huber", "UniversalKriging"]:
            result_table.append({
                "Model": model, "Param": Record[model]["Param"],
                "MSPE": f"{Record[model]['MSPE']:.4f}", "RMSE": f"{Record[model]['RMSE']:.4f}", "MAE": f"{Record[model]['MAE']:.4f}", "R2": f"{Record[model]['R2']:.4f}",
                "Time": f"{Record[model]['Time']:.2f}s"
            })

        df_res = pd.DataFrame(result_table)
        print("\n", df_res.to_markdown(index=False, tablefmt="github"), sep="")

        # Save results internally for the current hour
        for model in ["OLS_wendland", "OLS_sphere", "DeepKriging_wendland", "DeepKriging_wendland*", "DeepKriging_mrts", "DeepKriging_sphere", "DeepKriging_sphere_Huber", "UniversalKriging"]:
            experiment_results[model]["MSPE"].append(Record[model]["MSPE"])
            experiment_results[model]["RMSE"].append(Record[model]["RMSE"])
            experiment_results[model]["MAE"].append(Record[model]["MAE"])
            experiment_results[model]["R2"].append(Record[model]["R2"])

        # Clean up matrices before next repeat
        del Phi_wendland, max_Phi_sphere, max_Phi_mrts, location_data, location_data_norm
        K.clear_session()
        gc.collect()

        print(f"\n✅ Completed Hour {hour:02d} ('{variable}') - Repeat {repeat+1}/{repeat_experiment}")

    # ========================================================
    # Summary Table for the current Hour
    # ========================================================
    print("\n" + "="*80)
    print(f"📊 Summary of repeat experiments for Hour {hour:02d} ('{variable}')")
    print("="*80)
    print(f"Selected Best Orders: {best_orders}")
    print("="*80)

    avg_results = []
    for model in ["OLS_wendland", "OLS_sphere", "DeepKriging_wendland", "DeepKriging_wendland*", "DeepKriging_mrts", "DeepKriging_sphere", "DeepKriging_sphere_Huber", "UniversalKriging"]:
        metrics = experiment_results[model]
        avg_results.append({
            "Model": model,
            "MSPE": f"{np.mean(metrics['MSPE']):.2f}±{np.std(metrics['MSPE']):.2f}",
            "RMSE": f"{np.mean(metrics['RMSE']):.2f}±{np.std(metrics['RMSE']):.2f}",
            "MAE": f"{np.mean(metrics['MAE']):.2f}±{np.std(metrics['MAE']):.2f}",
            "R2": f"{np.mean(metrics['R2']):.2f}±{np.std(metrics['R2']):.2f}",
        })

    df_avg = pd.DataFrame(avg_results)
    print("\n", df_avg.to_markdown(index=False, tablefmt="github"), sep="")

    if avg_results:
        # Extract the float part of RMSE (everything before the ± symbol) to find the best model
        best_model = min(avg_results, key=lambda x: float(x["RMSE"].split("±")[0]))
        print(f"\n🏆 Best Model for Hour {hour:02d} ('{variable}'): {best_model['Model']} (RMSE: {best_model['RMSE']})")
        
    # Store df in overarching dictionary if you want to look at it later
    all_hours_summary[f"Hour_{hour:02d}"] = df_avg

print("\n" + "#"*80)
print("🎉 ALL 24 HOURS PROCESSED SUCCESSFULLY!")
print("#"*80)



################################################################################
🚀 STARTING DATASET: Hour 05 | Target Column: 'vec.4'
################################################################################

🏃 Hour 05 ('vec.4') - Repeat 1/2, Seed=1234


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.



Tuning order parameter for OLS_sphere
   Order      Val Loss     Test MSE    
   ----------------------------------
   50         10.6371      10.6522     
   100        9.0019       9.0806      
   200        6.5102       6.6467      
   500        4.3081       4.3323      
   700        3.6944       3.6901      
   1000       3.0564       3.0757      
   1400       2.4906       2.4631       *
   Best order: 1400

Tuning order parameter for DeepKriging_mrts


I0000 00:00:1772152456.561108   23958 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5592 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1772152458.092503   25296 service.cc:152] XLA service 0x55d68b4d7400 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772152458.092526   25296 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3060 Ti, Compute Capability 8.6
I0000 00:00:1772152458.398397   25296 cuda_dnn.cc:529] Loaded cuDNN version 90501
I0000 00:00:1772152460.462626   25296 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


   Order      Val Loss     Test MSE    
   ----------------------------------
   50         0.8218       0.8071      
   100        0.4593       0.4426      
   200        0.2973       0.2938      
   500        0.2201       0.2092      
   700        0.2078       0.2001       *
   1000       0.2085       0.1982      
   1400       0.2612       0.2414      
   Best order: 700

Tuning order parameter for DeepKriging_sphere
   Order      Val Loss     Test MSE    
   ----------------------------------
   50         0.2052       0.2026      
   100        0.1380       0.1348      
   200        0.1036       0.1052      
   500        0.0948       0.0925      
   700        0.0926       0.0886      
   1000       0.0867       0.0841       *
   1400       0.0916       0.0908      
   Best order: 1000

Tuning order parameter for DeepKriging_sphere_Huber
   Order      Val Loss     Test MSE    
   ----------------------------------
   50         0.0952       0.2006      
   100        0.0678   

2026-02-28 00:45:57.202891: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.09GiB (rounded to 1171200000)requested by op _EagerConst
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2026-02-28 00:45:57.238830: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:512] ***********************************************************_______**********___________________*____


InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.